# FusionOps v1: Baseline Training (No Hints)

Training an LLM to schedule ML computation graph operations without any prior knowledge or heuristic guidance. The agent receives four fixed tasks (Linear Chain, Diamond Graph, MatMul, K-Split) with 6-8 operations each, and must discover fusion and retention strategies purely through RL exploration. This run establishes the baseline: can an LLM figure out compiler scheduling from scratch?

## §1 — GPU & Disk Check

In [ ]:
import subprocess, shutil, sys
print("Python:", sys.version.split()[0])
print()
print("=== nvidia-smi ===")
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv"]).decode())
except Exception as e:
    print("nvidia-smi not available:", e)
print("=== Disk (/workspace) ===")
try:
    total, used, free = shutil.disk_usage("/workspace")
    print(f"Total: {total/1e9:.1f} GB | Free: {free/1e9:.1f} GB")
except:
    print("/workspace not available")

Python: 3.11.10

=== nvidia-smi ===
name, memory.total [MiB], memory.free [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 24210 MiB, 550.127.05

=== Disk (/workspace) ===
Total: 1320940.9 GB | Free: 290632.8 GB


## §2 — Install Dependencies

In [ ]:
import subprocess, sys

# Step 1: Upgrade typing_extensions FIRST (torch 2.10 needs TypeIs from >= 4.10)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "typing_extensions>=4.12.0"], check=True)

# Step 2: Install unsloth (handles its own deps)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)

# Step 3: Install extras
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "numpy"], check=True)

print("Installed. Checking versions...")
import torch, transformers, trl, peft
print(f"torch={torch.__version__} transformers={transformers.__version__} trl={trl.__version__} peft={peft.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Installed. Checking versions...


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torch=2.10.0+cu128 transformers=5.5.0 trl=0.24.0 peft=0.19.1
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


## §3 — Workspace Setup

In [ ]:
import os, pathlib

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
HF_CACHE = os.path.join(WORKSPACE, "hf_cache")
OUTPUT_DIR = os.path.join(WORKSPACE, "fusionops-training-output-v4")
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Workspace: {WORKSPACE}")
print(f"Output: {OUTPUT_DIR}")

Workspace: /workspace
Output: /workspace/fusionops-training-output-v4


## §4 — Clone FusionOps Repo

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/AnishaRoy5555/fusionops-env.git"
REPO_DIR = os.path.join(WORKSPACE, "fusionops-env")

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print(f"Repo at {REPO_DIR} — pulling latest")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Working dir: {os.getcwd()}")

Cloning https://github.com/AnishaRoy5555/fusionops-env.git


Cloning into '/workspace/fusionops-env'...


Working dir: /workspace/fusionops-env


## §5 — Imports & Seed

In [ ]:
from unsloth import FastLanguageModel

import json, math, random, re, time, warnings, logging
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Silence noisy warnings
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*max_new_tokens.*")
warnings.filterwarnings("ignore", message=".*use_return_dict.*")
try:
    import transformers as _tf
    _tf.logging.set_verbosity_error()
except:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seed: {SEED}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_1914/1632951082.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Seed: 42


## §6 — Config

In [ ]:
CONFIG = {
    "episodes": 300,
    "eval_interval": 50,
    "num_candidates": 4,
    "lr": 1e-5,
    "grad_accum": 4,
    "model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "output_dir": OUTPUT_DIR,
    "seed": SEED,
    "max_seq_length": 4096,
    "lora_r": 16,
    "lora_alpha": 32,
    "dpo_beta": 0.1,
    "max_steps_per_episode": 10,   # cap steps per episode for training
    "random_graph_ratio": 0.3,     # 30% of episodes use random graphs
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))

{
  "episodes": 300,
  "eval_interval": 50,
  "num_candidates": 4,
  "lr": 1e-05,
  "grad_accum": 4,
  "model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
  "output_dir": "/workspace/fusionops-training-output-v4",
  "seed": 42,
  "max_seq_length": 4096,
  "lora_r": 16,
  "lora_alpha": 32,
  "dpo_beta": 0.1,
  "max_steps_per_episode": 10,
  "random_graph_ratio": 0.3
}


## §7 — Environment + Random Graph Generator

In [ ]:
from src.tasks import load_task, get_task_config, list_tasks
from src.environment import FusionOpsEnv, parse_action
from src.models import Graph, OpType
from src.observation import _find_ready_ops, _generate_action_hints

print("Available tasks:", list_tasks())

def generate_random_graph(seed=None, num_ops_range=(4, 8)):
    """Generate a random valid computation graph for training diversity."""
    rng = random.Random(seed)
    num_ops = rng.randint(*num_ops_range)
    num_inputs = rng.randint(2, min(4, num_ops + 1))
    tensor_id = num_inputs
    widths = [128] * num_inputs
    heights = [128] * num_inputs
    op_inputs, op_outputs, op_types, base_costs = [], [], [], []
    available = list(range(num_inputs))

    for _ in range(num_ops):
        op_type = rng.choice(["Pointwise"] * 3 + ["MatMul"])
        if op_type == "MatMul" and len(available) >= 2:
            inp = rng.sample(available, 2)
            base_costs.append(rng.randint(1500, 3000))
        else:
            op_type = "Pointwise"
            n = rng.randint(1, min(2, len(available)))
            inp = rng.sample(available, n)
            base_costs.append(rng.randint(300, 1200))
        out_tid = tensor_id
        tensor_id += 1
        widths.append(128)
        heights.append(128)
        op_inputs.append(inp)
        op_outputs.append([out_tid])
        op_types.append(op_type)
        available.append(out_tid)

    return Graph.from_json({
        "widths": widths, "heights": heights, "inputs": op_inputs,
        "outputs": op_outputs, "base_costs": base_costs, "op_types": op_types,
        "fast_memory_capacity": 50000, "slow_memory_bandwidth": 10,
        "native_granularity": [128, 128],
    })

# Quick test
g = generate_random_graph(seed=42)
print(f"Random graph test: {len(g.operations)} ops, types={[op.op_type.value for op in g.operations]}")

Available tasks: ['task1_linear', 'task2_diamond', 'task3_matmul', 'task4_multistage']
Random graph test: 4 ops, types=['Pointwise', 'Pointwise', 'Pointwise', 'Pointwise']


## §8 — Load Model + LoRA

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Suppress max_length warning at source
model.generation_config.max_length = None
model.generation_config.max_new_tokens = None

FLM = FastLanguageModel
DEVICE = next(model.parameters()).device

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"Device: {DEVICE}")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.643 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Trainable: 29,933,568 / 1,728,606,208 (1.7%)
Device: cuda:0


## §9 — Prompts & Task Lists

In [ ]:
SYSTEM_PROMPT = """You are an agent interacting with an ML graph scheduling environment.

Each observation contains:
- LAST ACTION RESULT (only if your previous action was invalid - read this carefully)
- CURRENT STATE (completed ops, ready ops)
- VALID ACTION EXAMPLES (use these as templates)
- CONSTRAINTS and BEST PRACTICES

Your job:
- Pick ops only from the READY OPS list
- Follow the VALID ACTION EXAMPLES format exactly
- Goal: minimize total latency by fusing connected ops and reducing memory transfers

RESPOND WITH EXACTLY ONE LINE (no explanations):
SCHEDULE ops=[op_ids] config=[w,h,k] retain=[tensor_ids]"""

# Train on all 4 fixed tasks + random graphs
TRAIN_TASKS = ["task1_linear", "task2_diamond", "task3_matmul", "task4_multistage"]
# Eval on fixed tasks + a set of fixed-seed random graphs for generalization
EVAL_TASKS = ["task1_linear", "task2_diamond", "task3_matmul", "task4_multistage"]
EVAL_RANDOM_SEEDS = [1000, 1001, 1002]  # fixed seeds for reproducible eval

print("Training on:", TRAIN_TASKS, "+ random graphs")
print("Eval on:", EVAL_TASKS, "+ random seeds", EVAL_RANDOM_SEEDS)

Training on: ['task1_linear', 'task2_diamond', 'task3_matmul', 'task4_multistage'] + random graphs
Eval on: ['task1_linear', 'task2_diamond', 'task3_matmul', 'task4_multistage'] + random seeds [1000, 1001, 1002]


## §10 — Generation Helpers

In [ ]:
def extract_action(text: str) -> str:
    for line in text.split("\n"):
        line = line.strip()
        if "SCHEDULE" in line.upper() or "ops=" in line.lower():
            return line
    return text.strip() if text.strip() else "SCHEDULE ops=[0] config=[128,128,1] retain=[]"

def count_ops_in_action(action_text: str) -> int:
    match = re.search(r'ops\s*=\s*\[([^\]]*)\]', action_text)
    if match:
        inner = match.group(1).strip()
        if inner:
            return len([x for x in inner.split(",") if x.strip()])
    return 1

def generate_action(model, tokenizer, observation, temperature=0.7):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": observation + "\n\nChoose your next action:"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=80,
            temperature=max(temperature, 0.01),
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return extract_action(tokenizer.decode(generated, skip_special_tokens=True))

def generate_candidates(model, tokenizer, observation, num_candidates=4, temperature=0.8):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": observation + "\n\nChoose your next action:"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=80,
            temperature=max(temperature, 0.01),
            do_sample=True,
            num_return_sequences=num_candidates,
            pad_token_id=tokenizer.eos_token_id,
        )
    candidates = []
    for seq in outputs:
        gen = seq[inputs["input_ids"].shape[1]:]
        candidates.append(extract_action(tokenizer.decode(gen, skip_special_tokens=True)))
    return candidates

def get_temperature(episode, total_episodes):
    progress = episode / total_episodes
    if progress < 0.33: return 0.9
    elif progress < 0.66: return 0.7
    else: return 0.5

print("Generation helpers ready.")

Generation helpers ready.


## §11 — Multi-Step Episode Rollout (v4 core change)

The key difference from v3: we collect (observation, candidates, step_rewards) at EVERY step, not just the first. This trains the model on mid-episode decisions.

In [ ]:
def rollout_episode_multistep(graph, env, model, tokenizer, max_steps=10, temperature=0.7, num_candidates=4):
    """
    v4 multi-step rollout.
    At each step:
      1. Generate N candidates for the current observation
      2. Score each candidate by immediate step reward
      3. Play the best candidate
      4. Collect (obs, candidates, scores) for training

    Returns: (final_score, training_pairs, trajectory)
    training_pairs = [(observation, [candidates], [step_rewards]), ...]
    """
    result = env.reset()
    observation = result.observation

    training_pairs = []
    trajectory = []
    invalid_count = 0

    FLM.for_inference(model)

    for step in range(max_steps):
        if result.done:
            break

        # Generate candidates for THIS step's observation
        candidates = generate_candidates(model, tokenizer, observation, num_candidates, temperature)

        # Score each candidate by immediate step reward (not full episode)
        step_rewards = []
        step_infos = []
        for cand in candidates:
            # Create a temporary env copy to test this action
            from copy import deepcopy
            env_copy_state = deepcopy(env.state)

            action = parse_action(cand, graph)
            if action is None:
                step_rewards.append(-0.10)
                step_infos.append({"error": "parse_error"})
                continue

            # Temporarily apply the action to score it
            env_test = FusionOpsEnv(graph, max_steps=env.max_steps)
            env_test.state = deepcopy(env.state)
            env_test.state.step_count = env.state.step_count  # preserve step count
            test_result = env_test.step(action)
            step_rewards.append(test_result.reward)
            step_infos.append(test_result.info)

        # Collect training pair
        training_pairs.append((observation, candidates, step_rewards))

        # Play the best candidate in the real env
        best_idx = max(range(len(step_rewards)), key=lambda i: step_rewards[i])
        best_action = candidates[best_idx]

        action = parse_action(best_action, graph)
        if action is None:
            env.state.step_count += 1
            invalid_count += 1
            trajectory.append({"action": best_action, "reward": -0.10, "error": "parse_error"})
            if env.state.step_count >= env.max_steps:
                break
            # Use the observation as-is for next step
            continue

        result = env.step(action)
        error = result.info.get("error")
        if error:
            invalid_count += 1

        trajectory.append({
            "action": best_action,
            "reward": result.reward,
            "error": error,
        })
        observation = result.observation

        if result.done:
            break

    final_score = env.get_score()
    avg_fusion = sum(count_ops_in_action(t["action"]) for t in trajectory) / max(len(trajectory), 1)

    return final_score, training_pairs, trajectory, invalid_count, avg_fusion

# Quick test
g_test = load_task("task1_linear")
cfg_test = get_task_config("task1_linear")
env_test = FusionOpsEnv(g_test, max_steps=cfg_test["max_steps"])
score, pairs, traj, inv, fuse = rollout_episode_multistep(g_test, env_test, model, tokenizer, max_steps=5, num_candidates=2)
print(f"Test rollout: score={score:.3f}, training_pairs={len(pairs)}, steps={len(traj)}, invalid={inv}, fusion={fuse:.1f}")
for p in pairs:
    print(f"  candidates={[c[:40] for c in p[1]]}, rewards={[f'{r:.3f}' for r in p[2]]}")

Test rollout: score=0.083, training_pairs=5, steps=5, invalid=0, fusion=1.0
  candidates=['SCHEDULE ops=[0] config=[128,128,1] reta', 'SCHEDULE ops=[0] config=[128,128,1] reta'], rewards=['-0.080', '-0.080']
  candidates=['SCHEDULE ops=[1] config=[128,128,1] reta', 'SCHEDULE ops=[1] config=[128,128,1] reta'], rewards=['-0.080', '-0.080']
  candidates=['SCHEDULE ops=[2] config=[128,128,1] reta', 'SCHEDULE ops=[2] config=[128,128,1] reta'], rewards=['-0.080', '-0.080']
  candidates=['SCHEDULE ops=[3] config=[128,128,1] reta', 'SCHEDULE ops=[3] config=[128,128,1] reta'], rewards=['-0.080', '-0.080']
  candidates=['SCHEDULE ops=[4] config=[128,128,1] reta', 'SCHEDULE ops=[4] config=[128,128,1] reta'], rewards=['-0.080', '-0.080']


## §12 — GRPO Training Step (DPO-style, from v3)

In [ ]:
def _compute_logprob_impl(model, tokenizer, observation, action):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": observation[:2500] + "\n\nChoose your next action:"},
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + action

    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=3500)["input_ids"].to(DEVICE)

    prompt_len = prompt_ids.shape[1]
    if prompt_len >= full_ids.shape[1]:
        return (full_ids.sum() * 0.0).float()

    outputs = model(input_ids=full_ids)
    logits = outputs.logits
    shift_logits = logits[:, prompt_len - 1:-1, :]
    shift_labels = full_ids[:, prompt_len:]

    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
    return token_logp.sum()

def compute_logprob(model, tokenizer, observation, action):
    return _compute_logprob_impl(model, tokenizer, observation, action)

def compute_logprob_ref(model, tokenizer, observation, action):
    with torch.no_grad(), model.disable_adapter():
        return _compute_logprob_impl(model, tokenizer, observation, action).detach()

DPO_BETA = CONFIG["dpo_beta"]

def grpo_training_step(model, tokenizer, observation, candidates, scores):
    if len(candidates) < 2 or len(scores) < 2:
        return 0.0

    mean_score = sum(scores) / len(scores)
    advantages = [s - mean_score for s in scores]

    pairs = []
    for i in range(len(candidates)):
        for j in range(len(candidates)):
            if advantages[i] > advantages[j] + 0.001:
                pairs.append((candidates[i], candidates[j], advantages[i] - advantages[j]))

    if not pairs:
        return 0.0

    if len(pairs) > 6:
        pairs = sorted(pairs, key=lambda x: -x[2])[:6]

    FLM.for_training(model)
    model.train()

    total_loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)

    for better, worse, margin in pairs:
        logp_w = compute_logprob(model, tokenizer, observation, better)
        logp_l = compute_logprob(model, tokenizer, observation, worse)
        logp_w_ref = compute_logprob_ref(model, tokenizer, observation, better)
        logp_l_ref = compute_logprob_ref(model, tokenizer, observation, worse)

        dpo_logits = DPO_BETA * ((logp_w - logp_w_ref) - (logp_l - logp_l_ref))
        pair_loss = -torch.nn.functional.logsigmoid(dpo_logits) * margin
        total_loss = total_loss + pair_loss

    total_loss = total_loss / len(pairs)
    total_loss.backward()
    return total_loss.item()

print("GRPO training step ready (DPO-style with reference anchor).")

GRPO training step ready (DPO-style with reference anchor).


## §13 — Evaluation Helpers

In [ ]:
def evaluate_task(task_name, model, tokenizer, num_runs=3):
    scores = []
    for _ in range(num_runs):
        graph = load_task(task_name)
        cfg = get_task_config(task_name)
        env = FusionOpsEnv(graph, max_steps=cfg["max_steps"])
        result = env.reset()
        obs = result.observation

        FLM.for_inference(model)
        for step in range(cfg["max_steps"]):
            if result.done: break
            action_text = generate_action(model, tokenizer, obs, temperature=0.3)
            action = parse_action(action_text, graph)
            if action is None:
                env.state.step_count += 1
                if env.state.step_count >= cfg["max_steps"]: break
                continue
            result = env.step(action)
            obs = result.observation
            if result.done: break
        scores.append(env.get_score())
    return sum(scores) / len(scores)

def evaluate_random_graph(seed, model, tokenizer, num_runs=3):
    scores = []
    for _ in range(num_runs):
        graph = generate_random_graph(seed=seed)
        env = FusionOpsEnv(graph, max_steps=15)
        result = env.reset()
        obs = result.observation

        FLM.for_inference(model)
        for step in range(15):
            if result.done: break
            action_text = generate_action(model, tokenizer, obs, temperature=0.3)
            action = parse_action(action_text, graph)
            if action is None:
                env.state.step_count += 1
                if env.state.step_count >= 15: break
                continue
            result = env.step(action)
            obs = result.observation
            if result.done: break
        scores.append(env.get_score())
    return sum(scores) / len(scores)

def evaluate_all(model, tokenizer, num_runs=3):
    results = {}
    for task in EVAL_TASKS:
        results[task] = evaluate_task(task, model, tokenizer, num_runs)
    for seed in EVAL_RANDOM_SEEDS:
        results[f"random_{seed}"] = evaluate_random_graph(seed, model, tokenizer, num_runs)
    return results

def log_behavior(model, tokenizer):
    examples = []
    for task in EVAL_TASKS:
        graph = load_task(task)
        cfg = get_task_config(task)
        env = FusionOpsEnv(graph, max_steps=cfg["max_steps"])
        result = env.reset()
        obs = result.observation
        actions = []
        FLM.for_inference(model)
        for step in range(cfg["max_steps"]):
            if result.done: break
            a = generate_action(model, tokenizer, obs, temperature=0.3)
            actions.append(a)
            action = parse_action(a, graph)
            if action is None:
                env.state.step_count += 1
                if env.state.step_count >= cfg["max_steps"]: break
                continue
            result = env.step(action)
            obs = result.observation
            if result.done: break
        examples.append({"task": task, "score": env.get_score(), "actions": actions[:6]})
    return examples

print("Evaluation helpers ready.")

Evaluation helpers ready.


## §14 — Baseline Evaluation

In [ ]:
print("--- Baseline Evaluation (before training) ---")
baseline_scores = evaluate_all(model, tokenizer, num_runs=3)
for k, v in baseline_scores.items():
    print(f"  {k}: {v:.3f}")

baseline_behavior = log_behavior(model, tokenizer)
print("\nBaseline behavior:")
for b in baseline_behavior:
    print(f"  {b['task']}: score={b['score']:.3f} actions={b['actions'][:3]}")

--- Baseline Evaluation (before training) ---
  task1_linear: 0.000
  task2_diamond: 0.000
  task3_matmul: 0.051
  task4_multistage: 0.023
  random_1000: 0.057
  random_1001: 0.176
  random_1002: 0.025

Baseline behavior:
  task1_linear: score=0.000 actions=['SCHEDULE ops=[0] config=[128,128,1] retain=[]', 'SCHEDULE ops=[1] config=[128,128,1] retain=[]', 'SCHEDULE ops=[2] config=[128,128,1] retain=[]']
  task2_diamond: score=0.000 actions=['SCHEDULE ops=[0] config=[128,128,1] retain=[]', 'SCHEDULE ops=[1] config=[128,128,1] retain=[]', 'SCHEDULE ops=[2] config=[128,128,1] retain=[]']
  task3_matmul: score=0.000 actions=['SCHEDULE ops=[0] config=[128,128,1] retain=[]', 'SCHEDULE ops=[1] config=[128,128,1] retain=[]', 'SCHEDULE ops=[2] config=[128,128,1] retain=[]']
  task4_multistage: score=0.000 actions=['SCHEDULE ops=[0] config=[128,128,1] retain=[]', 'SCHEDULE ops=[1] config=[128,128,1] retain=[]', 'SCHEDULE ops=[2] config=[128,128,1] retain=[]']


## §15 — Training Loop (v4: multi-step + random graphs)

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG["lr"],
)

training_log = {
    "config": CONFIG,
    "baseline_scores": baseline_scores,
    "baseline_behavior": baseline_behavior,
    "episodes": [],
    "eval_history": [],
}

episode_scores = []
episode_losses = []
episode_fusion_ratios = []
per_task_history = {}
random_graph_counter = 0

print(f"--- Training ({CONFIG['episodes']} episodes, multi-step GRPO) ---")
start_time = time.time()

for episode in range(1, CONFIG["episodes"] + 1):
    temperature = get_temperature(episode, CONFIG["episodes"])

    # Decide: fixed task or random graph
    use_random = random.random() < CONFIG["random_graph_ratio"]

    if use_random:
        random_graph_counter += 1
        graph = generate_random_graph(seed=episode * 7 + random_graph_counter)
        task_name = f"random_ep{episode}"
        max_steps = min(CONFIG["max_steps_per_episode"], 15)
    else:
        task_name = random.choice(TRAIN_TASKS)
        graph = load_task(task_name)
        cfg = get_task_config(task_name)
        max_steps = min(CONFIG["max_steps_per_episode"], cfg["max_steps"])

    env = FusionOpsEnv(graph, max_steps=max_steps)

    # Multi-step rollout with candidate generation at every step
    final_score, training_pairs, trajectory, inv_count, avg_fusion = rollout_episode_multistep(
        graph, env, model, tokenizer,
        max_steps=max_steps,
        temperature=temperature,
        num_candidates=CONFIG["num_candidates"],
    )

    # GRPO update on ALL collected (obs, candidates, rewards) pairs
    total_ep_loss = 0.0
    num_updates = 0
    for obs, candidates, step_rewards in training_pairs:
        loss = grpo_training_step(model, tokenizer, obs, candidates, step_rewards)
        total_ep_loss += loss
        if loss > 0:
            num_updates += 1

    avg_loss = total_ep_loss / max(len(training_pairs), 1)

    # Gradient accumulation
    if episode % CONFIG["grad_accum"] == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    # Metrics
    episode_scores.append(final_score)
    episode_losses.append(avg_loss)
    episode_fusion_ratios.append(avg_fusion)

    task_label = "random" if use_random else task_name
    training_log["episodes"].append({
        "episode": episode,
        "task": task_label,
        "score": final_score,
        "loss": avg_loss,
        "temperature": temperature,
        "fusion_ratio": avg_fusion,
        "steps": len(trajectory),
        "training_pairs": len(training_pairs),
        "updates": num_updates,
    })

    # Progress
    if episode % 10 == 0 or episode <= 5:
        elapsed = time.time() - start_time
        avg_recent = sum(episode_scores[-10:]) / min(len(episode_scores), 10)
        avg_fuse = sum(episode_fusion_ratios[-10:]) / min(len(episode_fusion_ratios), 10)
        print(f"  Ep {episode:3d}/{CONFIG['episodes']} | {task_label:<16s} | "
              f"score={final_score:.3f} avg10={avg_recent:.3f} | "
              f"loss={avg_loss:.4f} temp={temperature:.1f} | "
              f"fusion={avg_fuse:.1f}ops updates={num_updates} | {elapsed:.0f}s")

    # Periodic eval
    if episode % CONFIG["eval_interval"] == 0:
        if episode % CONFIG["grad_accum"] != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        print(f"\n  --- Eval at episode {episode} ---")
        eval_scores = evaluate_all(model, tokenizer, num_runs=3)
        for k, v in eval_scores.items():
            print(f"    {k}: {v:.3f}")
            if k not in per_task_history:
                per_task_history[k] = []
            per_task_history[k].append((episode, v))

        behavior = log_behavior(model, tokenizer)
        training_log["eval_history"].append({
            "episode": episode,
            "scores": eval_scores,
            "behavior": behavior,
        })
        print()

total_time = time.time() - start_time
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
optimizer.zero_grad()

print(f"\n--- Training Complete ({total_time:.0f}s = {total_time/3600:.1f}h) ---")

--- Training (300 episodes, multi-step GRPO) ---
  Ep   1/300 | task1_linear     | score=0.000 avg10=0.000 | loss=0.0000 temp=0.9 | fusion=1.0ops updates=0 | 7s
  Ep   2/300 | task2_diamond    | score=0.000 avg10=0.000 | loss=0.0000 temp=0.9 | fusion=1.0ops updates=0 | 14s
  Ep   3/300 | random           | score=0.184 avg10=0.061 | loss=0.0000 temp=0.9 | fusion=1.2ops updates=0 | 18s
Unsloth: Will smartly offload gradients to save VRAM!
  Ep   4/300 | task1_linear     | score=0.091 avg10=0.069 | loss=0.0069 temp=0.9 | fusion=1.2ops updates=1 | 28s
  Ep   5/300 | task1_linear     | score=0.091 avg10=0.073 | loss=0.0071 temp=0.9 | fusion=1.2ops updates=1 | 36s
  Ep  10/300 | random           | score=0.013 avg10=0.064 | loss=0.2078 temp=0.9 | fusion=1.4ops updates=9 | 133s
  Ep  20/300 | task4_multistage | score=0.035 avg10=0.047 | loss=0.1209 temp=0.9 | fusion=1.5ops updates=2 | 251s
  Ep  30/300 | random           | score=0.091 avg10=0.047 | loss=0.0225 temp=0.9 | fusion=1.4ops updates=

## §16 — Final Evaluation

In [ ]:
print("--- Final Evaluation ---")
final_scores = evaluate_all(model, tokenizer, num_runs=5)

print(f"\n{'Task/Graph':<25s} {'Before':>8s} {'After':>8s} {'Change':>8s}")
print("-" * 50)
for k in sorted(final_scores.keys()):
    before = baseline_scores.get(k, 0.0)
    after = final_scores[k]
    change = after - before
    print(f"{k:<25s} {before:>8.3f} {after:>8.3f} {change:>+8.3f}")

final_behavior = log_behavior(model, tokenizer)
print("\nFinal behavior:")
for b in final_behavior:
    print(f"  {b['task']}: score={b['score']:.3f}")
    for a in b['actions'][:4]:
        print(f"    {a[:70]}")

--- Final Evaluation ---


## §17 — Save Outputs

In [ ]:
print(f"Saving to {CONFIG['output_dir']}")

# LoRA adapter
lora_path = os.path.join(CONFIG["output_dir"], "fusionops-lora")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"  Saved: fusionops-lora/")

# Scores
scores_data = {"baseline": baseline_scores, "final": final_scores,
    "improvement": {k: final_scores[k] - baseline_scores.get(k, 0) for k in final_scores}}
with open(os.path.join(CONFIG["output_dir"], "scores.json"), "w") as f:
    json.dump(scores_data, f, indent=2)

# Comparison
with open(os.path.join(CONFIG["output_dir"], "comparison.json"), "w") as f:
    json.dump({"baseline": baseline_behavior, "final": final_behavior}, f, indent=2)

# Training log
with open(os.path.join(CONFIG["output_dir"], "training_log.json"), "w") as f:
    json.dump(training_log, f, indent=2)

print("  All files saved.")

## §18 — Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("FusionOps GRPO v4 Training Results", fontsize=14, fontweight="bold")

# Reward curve
ax = axes[0, 0]
ax.plot(episode_scores, alpha=0.2, color="blue")
if len(episode_scores) >= 20:
    sm = [sum(episode_scores[max(0,i-20):i+1])/min(i+1,20) for i in range(len(episode_scores))]
    ax.plot(sm, color="blue", linewidth=2, label="Score (smoothed)")
ax.set_xlabel("Episode"); ax.set_ylabel("Score"); ax.set_title("Training Reward Curve")
ax.legend(); ax.grid(True, alpha=0.3)

# Per-task eval
ax = axes[0, 1]
for k in sorted(per_task_history.keys()):
    if per_task_history[k]:
        eps, sc = zip(*per_task_history[k])
        ls = "--" if "random" in k else "-"
        ax.plot(eps, sc, marker="o", label=k[:15], linestyle=ls, markersize=4)
ax.set_xlabel("Episode"); ax.set_ylabel("Score"); ax.set_title("Per-Task Scores")
ax.legend(fontsize=6); ax.grid(True, alpha=0.3)

# Fusion ratio
ax = axes[1, 0]
if len(episode_fusion_ratios) >= 20:
    sm_f = [sum(episode_fusion_ratios[max(0,i-20):i+1])/min(i+1,20) for i in range(len(episode_fusion_ratios))]
    ax.plot(sm_f, color="green", linewidth=2)
else:
    ax.plot(episode_fusion_ratios, color="green")
ax.set_xlabel("Episode"); ax.set_ylabel("Avg Ops/Action"); ax.set_title("Fusion Ratio")
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5); ax.grid(True, alpha=0.3)

# Before vs After
ax = axes[1, 1]
fixed_tasks = [t for t in EVAL_TASKS]
x = range(len(fixed_tasks))
w = 0.35
bv = [baseline_scores.get(t, 0) for t in fixed_tasks]
av = [final_scores.get(t, 0) for t in fixed_tasks]
ax.bar([i-w/2 for i in x], bv, w, label="Before", color="lightcoral")
ax.bar([i+w/2 for i in x], av, w, label="After", color="steelblue")
ax.set_xticks(list(x))
ax.set_xticklabels([t.replace("task","T").replace("_","\n") for t in fixed_tasks], fontsize=7)
ax.set_ylabel("Score"); ax.set_title("Before vs After"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plot_path = os.path.join(CONFIG["output_dir"], "reward_curve.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved: {plot_path}")

## §19 — Summary

In [ ]:
print("=" * 70)
print("TRAINING SUMMARY (v4: multi-step + random graphs)")
print("=" * 70)
print(f"Episodes: {CONFIG['episodes']} | Time: {total_time:.0f}s ({total_time/3600:.1f}h)")
print(f"Random graphs used: {random_graph_counter} ({100*random_graph_counter/CONFIG['episodes']:.0f}%)")

print(f"\nFixed Tasks:")
for t in EVAL_TASKS:
    b = baseline_scores.get(t, 0.0)
    a = final_scores.get(t, 0.0)
    status = "IMPROVED" if a > b + 0.01 else ("same" if abs(a-b) < 0.01 else "declined")
    print(f"  {t}: {b:.3f} -> {a:.3f} [{status}]")

print(f"\nRandom Graph Generalization:")
for seed in EVAL_RANDOM_SEEDS:
    k = f"random_{seed}"
    b = baseline_scores.get(k, 0.0)
    a = final_scores.get(k, 0.0)
    print(f"  seed={seed}: {b:.3f} -> {a:.3f}")

early_f = sum(episode_fusion_ratios[:20])/min(len(episode_fusion_ratios),20)
late_f = sum(episode_fusion_ratios[-20:])/min(len(episode_fusion_ratios),20)
print(f"\nFusion ratio: {early_f:.1f} (early) -> {late_f:.1f} (late)")

print(f"\nOutputs: {CONFIG['output_dir']}/")
print("=" * 70)